# 09 · Exploratory analysis E1: ranking versus operating point

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

**Run notebook 08 first** (it is finished). This notebook runs one *exploratory* analysis, planned after the
confirmatory results were seen and written into `docs/protocol/exploratory_analyses.md` (section E1) before it runs.
It is outside the Holm family, has no p-values and **cannot change any confirmatory decision**: H1a stays
inconclusive (provisional until H4), and H1b, H1c and H2 stay not supported.

**What it asks.** On CSE-CIC-IDS2018 and UNSW-NB15 the frozen random forest still ranks attacks above benign traffic
better than chance (PR-AUC about 0.24), yet its frozen threshold finds only 8.7 % and 31.3 % of the attacks. E1
measures where the transfer fails: in the ranking, in the threshold, or in particular attack families. For each
evaluation set it reports PR-AUC and ROC-AUC, recall at the frozen threshold, recall at the best threshold for a
false-positive rate of 0.1 %, 1 % and 5 % (**oracle**: chosen from the target labels, an upper bound no deployed
detector could reach), recall when alerting on the top 1 % and 5 % of rows (no labels needed), how many attacks
are tied with the lowest-scored benign traffic, and all of this per attack family, with 95 % block-bootstrap CIs.

**Nothing is re-scored or refitted.** It reads the per-row scores that notebook 08 saved in
`MyDrive/PhD-AI-Cyber/scores/` and the labels of the same rows. Before computing anything it checks that the
freeze is intact, that every scores file matches the SHA-256 in its record, and that the rows reproduce the
confirmatory counts exactly. If any check fails it stops and writes nothing: copy the error into the project thread.

**Before you start:** *Runtime → Change runtime type*: **High-RAM on**. No GPU is needed (a CPU runtime is enough;
a GPU runtime with High-RAM also works). CSE-CIC-IDS2018 has 45 million rows.

**Interruptions are safe.** Each evaluation set is written to Drive as soon as it finishes and is never computed
twice. After a disconnect, run step 1 again, then the step that was interrupted.

| Step | What it does | Approximate time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `… passed`, no failures) | 4 min |
| 2 | check the freeze, the confirmatory records and the saved scores (expected three `OK` lines) | 2 min |
| 3 | E1 on the CICIDS2017 test partition (both feature spaces, the in-domain reference) | 2–10 min |
| 4 | E1 on UNSW-NB15 | 5–20 min |
| 5 | E1 on CSE-CIC-IDS2018 (45 M rows, 5 seeds) | 30–120 min |
| 6 | the summary tables, final save | 1 min |

**Where the results go.** In the Drive copy of the repository (`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/`)
and on GitHub. There is nothing to move by hand.

* `results/exploratory/operating_point/summary.md`: **the result**, as tables for the paper and the thesis
* `results/exploratory/operating_point/<evaluation set>__<feature space>.json`: every figure in full (per seed,
  seed mean, 95 % CI, the checks)
* `results/exploratory/logs/<time>_operating_point.log`: run logs

**When it finishes:** use *File → Download → Download .ipynb* and send the finished notebook to the project thread.

## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `… passed` with no failures)

Always run this cell first, also after a disconnect. It loads the newest code.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import datetime as _dt, json, os, pathlib, shutil, subprocess, time

DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

# Git keeps its database on the Colab disk, never on Drive: Drive's file system can crash git while it unpacks
# downloads ("unpack-objects died of signal 7"). The files themselves stay in the Drive folder, so every result
# is in Drive the moment it is written, and the sync below copies it to GitHub.
GIT_DIR = '/content/phd-repo.git'
STATE = pathlib.Path(DRIVE_ROOT, 'code', '.phd_sync_state.json')    # commit the Drive folder last matched
os.environ['GIT_DIR'], os.environ['GIT_WORK_TREE'] = GIT_DIR, REPO_DIR
RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')
_trust_drive_records = True

def _mask(text):
    return text.replace(TOKEN or '\0', '***').strip()

def git(*args, check=False):
    r = subprocess.run(['git', *args], capture_output=True, text=True, cwd=REPO_DIR)
    out = _mask(r.stdout + r.stderr)
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def _stamp():
    return _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def _last_synced_commit():
    """The commit the Drive folder was last synced to: from the state file, else from the old .git on Drive."""
    try:
        if STATE.exists():
            return json.loads(STATE.read_text()).get('commit')
        g = pathlib.Path(REPO_DIR, '.git')
        head = (g / 'HEAD').read_text().strip()
        if not head.startswith('ref: '):
            return head
        ref = head[5:]
        if (g / ref).exists():
            return (g / ref).read_text().strip()
        for line in (g / 'packed-refs').read_text().splitlines():
            if line.endswith(' ' + ref):
                return line.split()[0]
    except (OSError, ValueError):
        pass
    return None

def setup_git():
    """Once per Colab session: download the git database to the Colab disk and line it up with the Drive folder."""
    global _trust_drive_records
    if pathlib.Path(GIT_DIR, 'HEAD').exists():
        return
    pathlib.Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('GIT_DIR', 'GIT_WORK_TREE')}
    r = subprocess.run(['git', 'clone', '--bare', '--quiet', REMOTE, GIT_DIR], capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(_mask(r.stdout + r.stderr))
        raise SystemExit('git clone failed (see message above)')
    git('config', 'core.bare', 'false', check=True)
    git('config', 'core.filemode', 'false')
    old = _last_synced_commit()
    known = bool(old) and git('cat-file', '-e', f'{old}^{{commit}}')[0] == 0
    git('reset', '-q', old if known else 'HEAD', check=True)      # sets the index only; Drive files untouched
    _trust_drive_records = known
    if not known:
        print('note: could not tell which commit the Drive folder last matched; GitHub wins for every tracked '
              'file, and Drive versions that differ are backed up first')
    dot = pathlib.Path(REPO_DIR, '.git')
    if dot.exists():
        dest = pathlib.Path(DRIVE_ROOT, 'backups', f'old_drive_git_{_stamp()}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(dot), str(dest))
        print(f'old git folder on Drive moved to {dest} (no longer used)')
    print('git database on the Colab disk; Drive folder last matched ' + (old[:7] if known else 'an unknown commit'))

def _push_hint(out):
    o = out.lower()
    if any(k in o for k in ('authentication failed', 'invalid username', 'invalid credentials', '403', 'permission')):
        return ('GitHub did not accept the token. If it has expired, create a new fine-grained token (repository '
                'phd-ai-threat-detection, Contents: Read and write) and save it as the Colab secret GITHUB_TOKEN.')
    if 'rejected' in o or 'fetch first' in o or 'non-fast-forward' in o:
        return 'GitHub had a newer commit; fetching it and trying again.'
    return 'GitHub or the network did not respond; trying again in a moment.'

def sync(message=None, strict=True, also=()):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    Results are on Drive before this runs. If GitHub cannot be reached, strict=False only warns: the next
    sync (the next step, or step 1 in a new session) saves the same files.
    `also` names other files changed here on purpose (e.g. docs/protocol/prior_exposure.md), saved like results.
    """
    global _trust_drive_records
    setup_git()
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    tries, out = 5, ''
    for attempt in range(1, tries + 1):
        code, out = git('fetch', '--quiet', REMOTE, 'main')
        if code == 0:
            base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
            changed = set(git('diff', '--name-only', base)[1].splitlines())
            untracked = set(git('ls-files', '--others', '--exclude-standard')[1].splitlines())
            # untracked non-record files survive the reset unless GitHub has a file at the same path
            untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
            changed |= untracked
            is_file = lambda f: pathlib.Path(REPO_DIR, f).is_file()
            keep = sorted(f for f in changed if is_file(f) and (f in also or f.startswith(RECORD_DIRS)
                          and (_trust_drive_records or f in untracked)))
            other = sorted(f for f in changed if f not in keep and is_file(f))
            stamp = _stamp()
            hold = pathlib.Path('/content/sync_hold') / stamp
            for f in keep:
                (hold / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
            if other:
                bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
                for f in other:
                    (bak / f).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
                print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
            git('reset', '--hard', '--quiet', 'FETCH_HEAD', check=True)
            for f in keep:
                dest = pathlib.Path(REPO_DIR, f)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(hold / f, dest)
            if keep:
                git('add', '--', *keep, check=True)
                if git('diff', '--cached', '--quiet')[0] != 0:
                    git('commit', '-q', '-m', message or 'Save results produced in Colab', check=True)
            code, out = git('push', '--quiet', REMOTE, 'HEAD:main')
            if code == 0:
                head = git('log', '--oneline', '-1')[1]
                STATE.write_text(json.dumps({'commit': git('rev-parse', 'HEAD')[1], 'utc': stamp}))
                _trust_drive_records = True
                print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
                return True
        print(f'GitHub refused the save (try {attempt} of {tries}): {out or "no message"}')
        print('  ' + _push_hint(out))
        if attempt < tries:
            time.sleep(min(60, 4 * 2 ** (attempt - 1)))
    msg = ('Could not save to GitHub. Nothing is lost: every result is in the Drive folder, and the next save '
           '(the next step, or step 1 after reopening the notebook) sends it to GitHub. Copy the messages above '
           'into the project thread if this keeps happening.')
    if strict:
        raise SystemExit(msg)
    print('WARNING: ' + msg)
    return False


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/evaluation/operating_point.py').exists(), 'old code: run this cell again'
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests
import sys
sys.path.insert(0, REPO_DIR)
from IPython.display import Markdown, display

def push(message, also=()):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records.
    If GitHub cannot be reached it only warns: the results are in Drive and the next save sends them."""
    if sync(message, strict=False, also=also):
        print('saved to GitHub:', message)

def explore(*args, message):
    """Run scripts/explore.py (its output shows below); push the records only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/explore.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')

## Step 2 · Check the freeze, the confirmatory records and the saved scores (expected: three `OK` lines)

Nothing is changed here. Expected: `OK Levels 1-2: random_forest …`, `OK confirmatory decisions …` and
`OK 20 scores files …`.

In [ ]:
import json, psutil
ram = psutil.virtual_memory().total / 2**30
print(f'RAM {ram:.0f} GB')
if ram < 40:
    print('WARNING: High-RAM seems to be off. Turn it on (Runtime -> Change runtime type) before step 5.')
from src.models import seal, score as S, develop as D
from src.evaluation import operating_point as E
f = seal.require_frozen(track='l12')
print(f"OK Levels 1-2: {f['proposed']}, frozen {f['frozen_utc']}; {len(f['model_files_sha256'])} model files and "
      f"{len(f['development_records_sha256'])} development records match their SHA-256")
dec = json.loads(pathlib.Path(REPO_DIR, 'results/confirmatory/decisions.json').read_text())
print('OK confirmatory decisions of ' + dec['generated_utc'] + ': ' +
      ', '.join(f"{h} {c['decision']}" for h, c in dec['decisions']['claims'].items() if c.get('estimate') is not None))
missing = [str(S.scores_path(e, sp, f['proposed'], s)) for e, sp in E.SETS for s in D.seeds()
           if not S.scores_path(e, sp, f['proposed'], s).exists()]
if missing:
    raise SystemExit('scores files missing (notebook 08 must have finished):\n' + '\n'.join(missing))
print(f'OK {len(E.SETS) * len(D.seeds())} scores files of the proposed detector found in Drive')
assert '## E1.' in pathlib.Path(REPO_DIR, 'docs/protocol/exploratory_analyses.md').read_text(), 'run step 1 again'
!python scripts/explore.py --status

## Step 3 · E1 on the CICIDS2017 test partition (2–10 min)

Expected: `[cicids2017_test] 221,066 rows read`, then 10 lines like `[cicids2017_test/cic] seed 11: … distinct scores, PR-AUC …`, two `written` lines and `in sync with GitHub at …`.

In [ ]:
explore('--operating-point', '--eval', 'cicids2017_test', message='Exploratory analysis E1 on the CICIDS2017 test partition')

## Step 4 · E1 on UNSW-NB15 (5–20 min)

Expected: `[unsw_nb15] 2,027,278 rows read`, 5 seed lines, one `written` line.

In [ ]:
explore('--operating-point', '--eval', 'unsw_nb15', message='Exploratory analysis E1 on UNSW-NB15')

## Step 5 · E1 on CSE-CIC-IDS2018 (30–120 min)

First it reads the labels of the 45 million rows from Drive (`[csecicids2018] 45,147,292 rows read`, 10–30 min), then one line per seed (5–20 min each) and one `written` line. If the runtime disconnects, run step 1 and this step again.

In [ ]:
explore('--operating-point', '--eval', 'csecicids2018', message='Exploratory analysis E1 on CSE-CIC-IDS2018')

## Step 6 · The summary tables (expected: four rows in the first table and `in sync with GitHub at …`)

Oracle columns use target labels and are upper bounds; budget columns use no labels. These figures are exploratory
and are reported as such.

In [ ]:
explore('--status', message='Exploratory analysis E1 status')
sync('Final save of exploratory analysis E1', strict=True)
display(Markdown(pathlib.Path(REPO_DIR, 'results/exploratory/operating_point/summary.md').read_text()))
print('On GitHub: https://github.com/sandesh20lamichhane/phd-ai-threat-detection/tree/main/results/exploratory')
print('Done. Download this notebook (File -> Download -> Download .ipynb) and send it to the project thread.')